# Dataset Exploration

Notebook generico per visualizzare dataset raw e output del progetto.

L'idea e semplice: cambi il `DATASET_PATH` nella cella di configurazione e puoi esplorare rapidamente `csv`, `parquet` o `xlsx`.

In [ ]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv

sns.set_theme(style='whitegrid')

project_root = Path.cwd()
if not (project_root / 'scripts').exists():
    project_root = project_root.parent

load_dotenv(project_root / '.env')
project_output_dir = Path(os.getenv('PROJECT_OUTPUT_DIR', '')) if os.getenv('PROJECT_OUTPUT_DIR') else None
local_output_dir = project_root / 'outputs' / 'local_preview'


## Configurazione

Scegli qui il file che vuoi esplorare. Puoi usare dati raw di BioCube oppure output prodotti da `selected_area_indicators.py`.

In [ ]:
# Esempi utili:
# DATASET_PATH = Path('/Volumes/Archivio/biomap_thesis/data/biocube/Species/europe_species.parquet')
# DATASET_PATH = local_output_dir / 'selected_milano_area_monthly.xlsx'
# DATASET_PATH = local_output_dir / 'selected_milano_cells.parquet'

DATASET_PATH = local_output_dir / 'selected_milano_area_monthly.xlsx'
N_ROWS = 10
MAX_NUMERIC_PLOTS = 4
MAP_SAMPLE_SIZE = 5000
DATASET_PATH


In [ ]:
def read_dataset(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == '.csv':
        return pd.read_csv(path)
    if suffix == '.parquet':
        return pd.read_parquet(path)
    if suffix in {'.xlsx', '.xls'}:
        return pd.read_excel(path)
    raise ValueError(f'Formato non supportato: {suffix}')


df = read_dataset(DATASET_PATH)
if 'month' in df.columns:
    df['month'] = pd.to_datetime(df['month'])

print('path:', DATASET_PATH)
print('shape:', df.shape)
df.head(N_ROWS)


In [ ]:
overview = pd.DataFrame({
    'column': df.columns,
    'dtype': [str(dtype) for dtype in df.dtypes],
    'missing_values': [int(df[col].isna().sum()) for col in df.columns],
    'missing_pct': [round(float(df[col].isna().mean() * 100), 2) for col in df.columns],
})
overview


In [ ]:
numeric_cols = df.select_dtypes(include='number').columns.tolist()
if numeric_cols:
    display(df[numeric_cols].describe().T)
else:
    print('Nessuna colonna numerica disponibile.')


In [ ]:
numeric_cols = df.select_dtypes(include='number').columns.tolist()
plot_cols = numeric_cols[:MAX_NUMERIC_PLOTS]

if plot_cols:
    fig, axes = plt.subplots(len(plot_cols), 1, figsize=(10, 4 * len(plot_cols)))
    if len(plot_cols) == 1:
        axes = [axes]
    for ax, column in zip(axes, plot_cols):
        sns.histplot(df[column].dropna(), ax=ax, kde=True)
        ax.set_title(f'Distribuzione di {column}')
    plt.tight_layout()
else:
    print('Nessuna colonna numerica da plottare.')


In [ ]:
if 'month' in df.columns:
    value_cols = [col for col in df.select_dtypes(include='number').columns if col not in {'latitude', 'longitude'}]
    if value_cols:
        monthly = df.sort_values('month')
        fig, axes = plt.subplots(len(value_cols[:3]), 1, figsize=(12, 4 * len(value_cols[:3])))
        if len(value_cols[:3]) == 1:
            axes = [axes]
        for ax, column in zip(axes, value_cols[:3]):
            sns.lineplot(data=monthly, x='month', y=column, ax=ax)
            ax.set_title(f'Andamento temporale di {column}')
        plt.tight_layout()
    else:
        print('Nessuna misura numerica adatta al grafico temporale.')
else:
    print('Colonna month non presente.')


In [ ]:
if {'latitude', 'longitude'}.issubset(df.columns):
    sample = df.sample(min(len(df), MAP_SAMPLE_SIZE), random_state=42)
    plt.figure(figsize=(8, 8))
    sns.scatterplot(data=sample, x='longitude', y='latitude', s=15, alpha=0.5)
    plt.title('Vista spaziale del dataset')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.show()
else:
    print('Coordinate latitude/longitude non presenti.')
